# ml_sandbox_31 — Random Forest (Light Grid Search)

Random Forest with light grid search instead of Optuna. Follows the nb21 pipeline structure:
SHAP importance → genre consolidation → forward selection → grid re-tune → threshold tuning.

**Grid:** 18 combinations over `max_depth` × `min_samples_leaf` × `max_features`.  
**vs nb21 RF:** No Optuna, same pipeline steps, same penalized selection criterion.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, log_loss,
    RocCurveDisplay, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.calibration import calibration_curve

In [ ]:
RANDOM_STATE = 42
N_SPLITS     = 5
LAM          = 0.3
MIN_N        = 5

PARAM_GRID = [
    {'n_estimators': n, 'max_depth': d, 'min_samples_leaf': l, 'max_features': f}
    for n, d, l, f in itertools.product([100, 300, 500], [3, 5, 8], [1, 5, 10], ['sqrt', 0.5])
]

df = pd.read_csv('../df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

imp         = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imp.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp  = pd.DataFrame(imp.transform(X_test),      columns=X_test.columns,  index=X_test.index)

print(f'Train: {X_train_imp.shape}  Test: {X_test_imp.shape}')
print(f'Class balance (train): {y_train.mean():.3f} hitmaker')
print(f'Grid size: {len(PARAM_GRID)} combinations')

In [ ]:
def build_rf(params):
    return RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params.get('max_depth'),
        min_samples_leaf=params.get('min_samples_leaf', 1),
        max_features=params.get('max_features', 'sqrt'),
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1,
    )


def cv_evaluate(X, y, params, skf):
    fold_val, fold_tr = [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_f, X_v = X.iloc[tr_idx], X.iloc[val_idx]
        y_f, y_v = y.iloc[tr_idx], y.iloc[val_idx]
        m = build_rf(params)
        m.fit(X_f, y_f)
        fold_val.append(roc_auc_score(y_v, m.predict_proba(X_v)[:, 1]))
        fold_tr.append(roc_auc_score(y_f,  m.predict_proba(X_f)[:, 1]))
    cv_auc = np.mean(fold_val)
    gap    = np.mean(fold_tr) - cv_auc
    return {'CV AUC': cv_auc, 'CV Std': np.std(fold_val),
            'Train AUC': np.mean(fold_tr), 'Overfit Gap': gap,
            'Penalized': cv_auc - LAM * gap}


def grid_search_rf(X, y, skf):
    best_params, best_score = None, -np.inf
    for params in PARAM_GRID:
        res = cv_evaluate(X, y, params, skf)
        if res['Penalized'] > best_score:
            best_score, best_params = res['Penalized'], params
    return best_params, best_score


def compute_shap_importance(model, X_val):
    try:
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_val)
        if isinstance(sv, list): sv = sv[1]
        elif sv.ndim == 3:       sv = sv[:, :, 1]
        return pd.DataFrame({'Feature': X_val.columns,
                             'Importance': np.abs(sv).mean(axis=0)})\
                 .sort_values('Importance', ascending=False).reset_index(drop=True)
    except Exception:
        perm = permutation_importance(model, X_val, n_repeats=10,
                   random_state=RANDOM_STATE, scoring='roc_auc')
        return pd.DataFrame({'Feature': X_val.columns,
                             'Importance': perm.importances_mean})\
                 .sort_values('Importance', ascending=False).reset_index(drop=True)

## Step 1 — Grid Search (18 combinations)

Best params selected by penalized CV AUC: `CV AUC − λ × Overfit Gap`.

In [ ]:
print(f'Grid search ({len(PARAM_GRID)} combinations)...')
params_full, best_score = grid_search_rf(X_train_imp, y_train, skf)
res_full = cv_evaluate(X_train_imp, y_train, params_full, skf)

print(f'  Best params : {params_full}')
print(f'  CV AUC      : {res_full["CV AUC"]:.4f} ± {res_full["CV Std"]:.4f}')
print(f'  Train AUC   : {res_full["Train AUC"]:.4f}')
print(f'  Gap         : {res_full["Overfit Gap"]:.4f}')
print(f'  Penalized   : {res_full["Penalized"]:.4f}')

## Step 2 — CV SHAP Importance + Genre Consolidation

In [ ]:
fold_imps = []
for tr_idx, val_idx in skf.split(X_train_imp, y_train):
    X_f, X_v = X_train_imp.iloc[tr_idx], X_train_imp.iloc[val_idx]
    y_f, y_v = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    m = build_rf(params_full)
    m.fit(X_f, y_f)
    perm_df = compute_shap_importance(m, X_v)
    fold_imps.append(perm_df.set_index('Feature')['Importance'])

perm_df_full = (pd.concat(fold_imps, axis=1)
                  .mean(axis=1)
                  .sort_values(ascending=False)
                  .reset_index())
perm_df_full.columns = ['Feature', 'Importance']
print('Top 15 features by CV SHAP:')
print(perm_df_full.head(15).to_string(index=False))

# Genre consolidation
all_genre_cols = [c for c in X_train_imp.columns if c.startswith('artist_genre_')]
genre_imp      = perm_df_full.set_index('Feature')['Importance']
genre_vals     = genre_imp[[c for c in all_genre_cols if c in genre_imp.index]]
high_genres    = [c for c in all_genre_cols if genre_imp.get(c, 0) > genre_vals.mean()]
low_genres     = [c for c in all_genre_cols if c not in high_genres]

X_tr_c = X_train_imp.drop(columns=low_genres).copy()
X_te_c = X_test_imp.drop(columns=low_genres).copy()
if low_genres:
    X_tr_c['artist_genre_other'] = (X_train_imp[low_genres].sum(axis=1) > 0).astype(int)
    X_te_c['artist_genre_other'] = (X_test_imp[low_genres].sum(axis=1) > 0).astype(int)

print(f'\nGenres kept  : {[c.replace("artist_genre_","") for c in high_genres]}')
print(f'Folded into other: {[c.replace("artist_genre_","") for c in low_genres]}')
print(f'Consolidated features: {X_tr_c.shape[1]}')

## Step 3 — Forward Selection

Greedy forward selection ordered by CV SHAP importance. `n_optimal` selected by peak CV AUC.

In [ ]:
perm_cons_order = perm_df_full.set_index('Feature')['Importance']
feature_order   = [f for f in perm_cons_order.sort_values(ascending=False).index
                   if f in X_tr_c.columns]
if 'artist_genre_other' in X_tr_c.columns and 'artist_genre_other' not in feature_order:
    feature_order.append('artist_genre_other')

sel_rows = []
for n in range(min(3, len(feature_order)), len(feature_order) + 1):
    feats = feature_order[:n]
    res   = cv_evaluate(X_tr_c[feats], y_train, params_full, skf)
    sel_rows.append({'n_features': n,
                     'CV AUC':    round(res['CV AUC'], 4),
                     'Train AUC': round(res['Train AUC'], 4),
                     'Gap':       round(res['Overfit Gap'], 4),
                     'Penalized': round(res['Penalized'], 4)})

df_sel = pd.DataFrame(sel_rows).set_index('n_features')
print(df_sel.to_string())

n_optimal = int(df_sel['CV AUC'].idxmax())
n_optimal = max(n_optimal, MIN_N)
feats_sel = feature_order[:n_optimal]
print(f'\nn_optimal = {n_optimal}  →  {feats_sel}')

## Step 4 — Grid Re-tune on Selected Features

In [ ]:
print(f'Grid search on n={n_optimal} features ({len(PARAM_GRID)} combinations)...')
params_final, _ = grid_search_rf(X_tr_c[feats_sel], y_train, skf)
res_final = cv_evaluate(X_tr_c[feats_sel], y_train, params_final, skf)

print(f'  Best params : {params_final}')
print(f'  CV AUC      : {res_final["CV AUC"]:.4f} ± {res_final["CV Std"]:.4f}')
print(f'  Train AUC   : {res_final["Train AUC"]:.4f}')
print(f'  Gap         : {res_final["Overfit Gap"]:.4f}')
print(f'  Penalized   : {res_final["Penalized"]:.4f}')

feats_final = feats_sel

# Fit final model
model_final = build_rf(params_final)
model_final.fit(X_tr_c[feats_final], y_train)

y_proba_te = model_final.predict_proba(X_te_c[feats_final])[:, 1]
y_proba_tr = model_final.predict_proba(X_tr_c[feats_final])[:, 1]

test_auc  = roc_auc_score(y_test, y_proba_te)
train_auc = roc_auc_score(y_train, y_proba_tr)
gap       = train_auc - test_auc

print(f'\nTest evaluation:')
print(f'  Test AUC  : {test_auc:.4f}')
print(f'  Train AUC : {train_auc:.4f}')
print(f'  Gap       : {gap:.4f}')
print(f'  Logloss   : {log_loss(y_test, y_proba_te):.4f}')
print(f'  Brier     : {brier_score_loss(y_test, y_proba_te):.4f}')

## Step 5 — Threshold Tuning

OOF probabilities used to tune threshold — avoids test set leakage.

In [ ]:
oof_proba = np.zeros(len(y_train))
for tr_idx, val_idx in skf.split(X_tr_c[feats_final], y_train):
    X_f, X_v = X_tr_c[feats_final].iloc[tr_idx], X_tr_c[feats_final].iloc[val_idx]
    y_f      = y_train.iloc[tr_idx]
    m = build_rf(params_final)
    m.fit(X_f, y_f)
    oof_proba[val_idx] = m.predict_proba(X_v)[:, 1]

thresholds = np.arange(0.10, 0.90, 0.01)
rows = []
for t in thresholds:
    y_pred = (oof_proba >= t).astype(int)
    rows.append({
        'Threshold': round(t, 2),
        'Precision': precision_score(y_train, y_pred, zero_division=0),
        'Recall':    recall_score(y_train, y_pred, zero_division=0),
        'F1':        f1_score(y_train, y_pred, zero_division=0),
    })

df_thresh = pd.DataFrame(rows).set_index('Threshold')
best_t    = df_thresh['F1'].idxmax()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_thresh.index, df_thresh['Precision'], label='Precision')
ax.plot(df_thresh.index, df_thresh['Recall'],    label='Recall')
ax.plot(df_thresh.index, df_thresh['F1'],        label='F1', lw=2)
ax.axvline(best_t, color='black', linestyle='--', label=f'Best F1 @ {best_t}')
ax.axvline(0.5,    color='grey',  linestyle=':',  label='Default 0.5')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('OOF Threshold Tuning — Random Forest')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Best OOF threshold: {best_t}  (F1={df_thresh.loc[best_t, "F1"]:.3f})')
for t in [0.5, best_t]:
    y_pred = (y_proba_te >= t).astype(int)
    print(f'Threshold={t}:  P={precision_score(y_test, y_pred):.3f}  '
          f'R={recall_score(y_test, y_pred):.3f}  F1={f1_score(y_test, y_pred):.3f}')

## Diagnostics

In [ ]:
y_pred_final = (y_proba_te >= best_t).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'Random Forest — Diagnostics  (n={len(feats_final)}, threshold={best_t})',
             fontsize=13, fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final, ax=axes[0][0],
    display_labels=['Non-Hitmaker', 'Hitmaker'], colorbar=False
)
axes[0][0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(y_test, y_proba_te, ax=axes[0][1],
                                  name=f'RF (AUC={test_auc:.3f})', color='#55A868')
axes[0][1].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0][1].set_title('ROC Curve')

prob_true, prob_pred = calibration_curve(y_test, y_proba_te, n_bins=10)
axes[1][0].plot(prob_pred, prob_true, 's-', color='#55A868', label='RF')
axes[1][0].plot([0,1],[0,1],'k--',alpha=0.4,label='Perfect')
axes[1][0].set_xlabel('Mean predicted probability')
axes[1][0].set_ylabel('Fraction of positives')
axes[1][0].set_title('Calibration Curve')
axes[1][0].legend(); axes[1][0].grid(True, alpha=0.3)

axes[1][1].hist(y_proba_te[y_test == 0], bins=20, alpha=0.6,
                color='steelblue', label='Non-Hitmaker')
axes[1][1].hist(y_proba_te[y_test == 1], bins=20, alpha=0.6,
                color='#55A868', label='Hitmaker')
axes[1][1].axvline(best_t, color='black', linestyle='--', label=f'Threshold={best_t}')
axes[1][1].set_xlabel('Predicted probability')
axes[1][1].set_ylabel('Count')
axes[1][1].set_title('Score Distribution')
axes[1][1].legend(); axes[1][1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Feature Importance (SHAP)

In [ ]:
explainer = shap.TreeExplainer(model_final)
sv = explainer.shap_values(X_tr_c[feats_final])
if isinstance(sv, list): sv = sv[1]
elif sv.ndim == 3:        sv = sv[:, :, 1]

shap_abs    = pd.Series(np.abs(sv).mean(axis=0), index=feats_final).sort_values(ascending=False)
shap_signed = pd.Series(sv.mean(axis=0), index=feats_final).reindex(shap_abs.index)
colors = ['#55A868' if v >= 0 else '#C44E52' for v in shap_signed]

fig, ax = plt.subplots(figsize=(8, max(4, len(feats_final) * 0.45)))
ax.barh(shap_abs.index[::-1], shap_abs.values[::-1], color=colors[::-1])
ax.set_xlabel('Mean |SHAP|')
ax.set_title('Random Forest — Feature Importance\n'
             'Green = pushes toward Hitmaker, Red = toward Non-Hitmaker')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

print('\nSummary:')
for f in shap_abs.index:
    direction = '→ Hitmaker' if shap_signed[f] >= 0 else '→ Non-Hitmaker'
    print(f'  {f:<45}  SHAP={shap_abs[f]:.4f}  {direction}')

## Final Summary

In [ ]:
y_pred_50  = (y_proba_te >= 0.50).astype(int)
y_pred_opt = (y_proba_te >= best_t).astype(int)

print('=' * 60)
print('  Random Forest — Final Results (nb31)')
print('=' * 60)
print(f'  N features  : {len(feats_final)}')
print(f'  Features    : {feats_final}')
print(f'  Best params : {params_final}')
print(f'  λ           : {LAM}')
print()
print(f'  CV AUC    : {res_final["CV AUC"]:.4f} ± {res_final["CV Std"]:.4f}')
print(f'  Train AUC : {train_auc:.4f}')
print(f'  Test AUC  : {test_auc:.4f}')
print(f'  Gap       : {gap:.4f}')
print(f'  Logloss   : {log_loss(y_test, y_proba_te):.4f}')
print(f'  Brier     : {brier_score_loss(y_test, y_proba_te):.4f}')
print()
print(f'  Threshold=0.50  P={precision_score(y_test,y_pred_50):.3f}  '
      f'R={recall_score(y_test,y_pred_50):.3f}  F1={f1_score(y_test,y_pred_50):.3f}')
print(f'  Threshold={best_t}  P={precision_score(y_test,y_pred_opt):.3f}  '
      f'R={recall_score(y_test,y_pred_opt):.3f}  F1={f1_score(y_test,y_pred_opt):.3f}')
print()
print(f'  nb21 RF reference  →  Test AUC=0.7671  Gap=0.005  N=9')
print(f'  nb29 fixed-param   →  Test AUC mean=0.7602  (B=100)')